DSPy 是一个由斯坦福大学推出的框架，旨在将提示工程（Prompt Engineering）转变为编程。它的核心思想是：你定义签名（Signature）和逻辑结构（Module），而让 DSPy 的编译器（Compiler）去自动为你优化提示词。

安装 dspy:

In [1]:
%pip install dspy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install python-dotenv
import dotenv
dotenv.load_dotenv()


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


True

In [11]:
import os
api_key = os.getenv("GEMINI_API_KEY")
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

在使用 DSPy 之前，我们需要导入依赖:

In [4]:
import dspy

In [15]:
lm = dspy.LM(
    "openai/gemini-3-flash-preview", api_base=base_url, api_key=api_key, temperature=0
)
dspy.configure(lm=lm)

`dspy.configure(lm=lm)` 是在做什么？

简单来说这是 DSPy 的注册机制，作用是设置全局默认语言模型，如果没有显式指定模型，它们会自动寻找全局配置中的 `lm`。

在 DSPy 中，`dspy.configure` 和 `dspy.context` 都用于管理模型、回调函数等配置，区别是他们的使用场景不同。

`dspy.configure` 是当你希望整个脚本或应用默认使用某一个模型时使用。通常出现在代码的初始化阶段。

`dspy.context` 是局部作用域，仅在 `with` 语句块内有效，例如在多模型流水线，涉及线程安全的场景下使用，例如集成在 LangFlow 的组件。

## Example1: Basic Text Classification
> One input, one output field.

定义签名:

In [ ]:
class Sentiment(dspy.Signature):
    """Classify the sentiment of a given sentence."""

    sentence: str = dspy.InputField(default="I love this movie, it was fantastic!", desc="The sentence to analyze.")
    sentiment: str = dspy.OutputField(desc="positive, negative, or neutral")

In [21]:
Sentiment.input_fields

{'sentence': FieldInfo(annotation=str, required=False, default='I love this movie, it was fantastic!', json_schema_extra={'desc': 'The sentence to analyze.', '__dspy_field_type': 'input', 'prefix': 'Sentence:'})}

In [22]:
Sentiment.output_fields

{'sentiment': FieldInfo(annotation=str, required=True, json_schema_extra={'desc': 'positive, negative, or neutral', '__dspy_field_type': 'output', 'prefix': 'Sentiment:'})}

In [16]:
program = dspy.Predict(Sentiment)
pred = program(sentence="I love this movie, it was fantastic!")
pred["sentiment"]

'positive'